# MTL — Colab/Kaggle GPU training

Run cells top to bottom. This trains the shared ResNet50+FPN backbone
jointly on detection (RetinaNet) + semantic segmentation (FCN) +
multi-label classification, on a COCO subset.


## 1. Install dependencies
Colab usually ships a CUDA-matched torch/torchvision already — only reinstall if missing.

In [1]:
import torch
print(torch.__version__, torch.cuda.is_available())

# Uncomment only if the above shows no CUDA:
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

# timm: DINO ViT-B/16 omurgası için (configs/train_colab_dino.yaml). torch/torchvision'a dokunmaz.
!pip install pycocotools PyYAML tqdm scikit-learn requests timm

2.11.0+cu128 True


## 2. Get the repo
Either clone from git, or upload a zip of this project via the Colab file browser and unzip it.

In [2]:
# dino-backbone dalını klonla (DINO kodu + bu notebook bu dalda).
# main'e merge ettikten sonra `-b dino-backbone`'u kaldırabilirsin.
!git clone -b dino-backbone https://github.com/itu-itis23-ucgun22/mtl.git
%cd mtl
!pip install -e .

Cloning into 'mtl'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 110 (delta 36), reused 101 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 329.95 KiB | 8.05 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/mtl
Obtaining file:///content/mtl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mtl (pyproject.toml) ... done
  Created wheel for mtl: filename=mtl-0.1.0-0.editable-py3-none-any.whl size=1180 sha256=6e18b4cfaeb1d097905cce66463224d7803546477d2146c9e2f4d94d617481b8
  Stored in directory: /tmp/pip-ephem-wheel-cache-d6vp4y56/wheels/b8/3b/63/cd9bbc919e23d84900281b7352d08ba8b6890722307e161497
Successfully built mtl


## 2.5. Google Drive'ı bağla (veri + checkpoint'i kalıcı yap)
COCO subset'i **ve** `checkpoints/`'i Drive'da tutar (ikisi de symlink olur). Veri: ilk oturumda section 3 indirir, sonrakilerde Drive'dan gelir — baştan indirmez. Checkpoint: `train.py`'nin yazdığı `.pt` dosyaları doğrudan Drive'a düşer, runtime kopunca kaybolmaz; sonraki oturumda `--resume` ile devam edilebilir.

In [ ]:
# --- Google Drive: COCO subset'i kalıcı yap (her oturumda baştan indirmemek için) ---
from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil

# ResNet denemesinde veriyi zaten Drive'a koymuştun -> otomatik bul, yeniden indirme yok.
# (Yolunu biliyorsan aramayı atlamak için DRIVE_ROOT'u doğrudan o klasöre ayarla.)
DRIVE_ROOT = '/content/drive/MyDrive/mtl_data/coco_subset/annotations'

default = f'{DRIVE_ROOT}/coco_subset/annotations/instances_train_subset.json'
if os.path.exists(default):
    target = f'{DRIVE_ROOT}/coco_subset'
    print('Drive\'da mevcut veri (varsayılan konum):', target)
else:
    # Tüm MyDrive'ı tarayıp subset annotasyonunu bul (bir kerelik, biraz sürebilir).
    hits = glob.glob('/content/drive/MyDrive/**/instances_train_subset.json', recursive=True)
    if hits:
        target = os.path.dirname(os.path.dirname(hits[0]))  # .../coco_subset
        print('Drive\'da mevcut veri bulundu:', target)
    else:
        target = f'{DRIVE_ROOT}/coco_subset'
        os.makedirs(target, exist_ok=True)
        print('Drive\'da veri yok; ilk indirme buraya yazacak:', target)

# data/coco_subset -> Drive'daki klasöre symlink
os.makedirs('data', exist_ok=True)
link = 'data/coco_subset'
if os.path.islink(link):
    os.remove(link)
elif os.path.isdir(link):
    shutil.rmtree(link)
os.symlink(target, link)
print('coco_subset ->', os.path.realpath(link))

# --- checkpoint'ler de Drive'da kalsın ki oturum kesilince kaybolmasın ---
# checkpoints/ -> Drive'daki kalıcı klasöre symlink. Böylece train.py'nin yazdığı
# _epoch*.pt / _step*.pt dosyaları doğrudan Drive'a düşer; runtime kopunca kaybolmaz
# ve sonraki oturumda --resume ile kaldığın yerden devam edebilirsin.
CKPT_DRIVE = '/content/drive/MyDrive/mtl_data/checkpoints'
os.makedirs(CKPT_DRIVE, exist_ok=True)
ckpt_link = 'checkpoints'
if os.path.islink(ckpt_link):
    os.remove(ckpt_link)
elif os.path.isdir(ckpt_link):
    # Bu oturumda yerelde birikmiş checkpoint varsa önce Drive'a taşı, sonra symlink'e çevir.
    for f in glob.glob(f'{ckpt_link}/*'):
        dst = os.path.join(CKPT_DRIVE, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.move(f, dst)
    shutil.rmtree(ckpt_link)
os.symlink(CKPT_DRIVE, ckpt_link)
print('checkpoints ->', os.path.realpath(ckpt_link))
print('  mevcut:', sorted(os.listdir(CKPT_DRIVE)) or '(bos)')

## 3. COCO subset'i hazırla
İlk oturumda full COCO annotations'ı indirip subset'i (22.5k train / 2k val) oluşturur ve görüntüleri Drive'a indirir. Section 2.5 sayesinde **veri zaten Drive'daysa bu hücre indirmeyi atlar.**

In [13]:
%%bash
set -e
# Veri Drive'da zaten varsa (section 2.5 symlink'i) tüm indirmeyi atla.
if [ -f data/coco_subset/annotations/instances_train_subset.json ] \
   && [ -f data/coco_subset/annotations/instances_val_subset.json ] \
   && [ -n "$(ls -A data/coco_subset/images/train 2>/dev/null)" ] \
   && [ -n "$(ls -A data/coco_subset/images/val 2>/dev/null)" ]; then
  echo "Veri Drive'da mevcut -> indirme atlanıyor."
  exit 0
fi

# Full COCO annotations (subset JSON'u üretmek için gerekli) - subset zaten varsa buraya girilmez.
wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
unzip -q -o annotations_trainval2017.zip

python scripts/prepare_coco_subset.py \
    --ann-file annotations/instances_train2017.json \
    --out data/coco_subset/annotations/instances_train_subset.json \
    --n-images 22500
python scripts/prepare_coco_subset.py \
    --ann-file annotations/instances_val2017.json \
    --out data/coco_subset/annotations/instances_val_subset.json \
    --n-images 2000

# Sadece subset'in görüntülerini indir (script mevcut dosyaları atlar -> Drive'da varsa hızlı geçer).
python scripts/download_subset_images.py \
    --ann-file data/coco_subset/annotations/instances_train_subset.json \
    --out-dir data/coco_subset/images/train
python scripts/download_subset_images.py \
    --ann-file data/coco_subset/annotations/instances_val_subset.json \
    --out-dir data/coco_subset/images/val

loading annotations into memory...
Done (t=12.58s)
creating index...
index created!
Wrote 22500 images to data/coco_subset/annotations/instances_train_subset.json
loading annotations into memory...
Done (t=0.54s)
creating index...
index created!
Wrote 2000 images to data/coco_subset/annotations/instances_val_subset.json
Done. 22500/22500 images present in data/coco_subset/images/train.
Done. 2000/2000 images present in data/coco_subset/images/val.


downloading images: 100%|██████████| 2000/2000 [00:59<00:00, 33.79it/s]


## 3.5. Veriyi yerel diske kopyala (eğitim I/O'sunu hızlandır)
Google Drive mount'u (FUSE) binlerce küçük görüntüyü **rastgele** okumakta çok yavaştır (dosya başına ağ gecikmesi) → GPU veri beklerken boşta kalır, bütçe yanar. Bu hücre subset'i bir kez Colab'ın hızlı yerel SSD'sine (`/content`) kopyalar ve `data/coco_subset` symlink'ini oraya yöneltir. Config'ler değişmeden tüm eğitim/eval yerelden okur; Drive orijinali kalıcı depo olarak durur. **Section 3'ten (veri Drive'da hazır) sonra çalıştır.**

In [ ]:
# --- Veriyi Drive'dan yerel diske (/content) kopyala + symlink'i oraya çevir ---
# /content her oturumda silindiği için bu hücre her oturumda bir kez kopyalar.
import os, shutil, time

src = os.path.realpath('data/coco_subset')   # 2.5 hücresinin kurduğu Drive hedefi
LOCAL = '/content/coco_local'

if not os.path.exists(f'{LOCAL}/annotations/instances_train_subset.json'):
    print(f'Yerel diske kopyalanıyor: {src} -> {LOCAL} (bir kerelik, biraz sürebilir)...')
    t = time.time()
    shutil.copytree(src, LOCAL, dirs_exist_ok=True)
    print(f'Kopyalama bitti: {time.time() - t:.0f} sn')
else:
    print('Yerel kopya zaten var:', LOCAL)

# data/coco_subset symlink'ini Drive yerine yerel kopyaya çevir (config'ler değişmeden yerelden okur)
link = 'data/coco_subset'
if os.path.islink(link):
    os.remove(link)
elif os.path.isdir(link):
    shutil.rmtree(link)
os.symlink(LOCAL, link)
print('coco_subset ->', os.path.realpath(link), '(yerel disk; eğitim buradan okuyacak)')

## 4. Train

### 4.0. (Opsiyonel) GPU kullanımını ölç — I/O darboğazı var mı?
Kısa bir eğitimi arka planda koşup `nvidia-smi` ile GPU kullanımını basar. **Optimizasyon öncesi/sonrası** bir kez koşup kıyasla: `sm` sütunu (GPU hesap %) steady-state'te (~50. adımdan sonra) **%85-100 → darboğaz yok**; sık sık **%0-40'a düşüyorsa → GPU veri bekliyor** (3.5 yerel-kopya hücresini çalıştır). İlk ~45 sn ısınmadır (ağırlık indirme + worker açılışı), ona bakma.

In [ ]:
# --- GPU-util ölçümü: 150 adımlık kısa eğitimi arka planda koş, nvidia-smi ile izle ---
# Amaç metrik değil, GPU meşgul mü diye bakmak. checkpoint_every_steps>150 verip ara
# checkpoint yazmasını da engelliyoruz (bütçe/çöp dosya olmasın).
import subprocess, time

p = subprocess.Popen(
    "python scripts/train.py --config configs/train_colab_gpu.yaml "
    "--overrides train.max_steps=150 train.checkpoint_every_steps=1000",
    shell=True,
)
time.sleep(45)                    # ısınmayı (ağırlık indirme, worker açılışı) geç
!nvidia-smi dmon -s u -c 30       # 30 sn boyunca GPU(sm)/bellek(mem) kullanımı, saniyede bir satır
p.wait()
print("\n[ölçüm bitti] 'sm' sütununa bak: yüksek+sabit = iyi, sık düşüş = veri bekliyor.")

In [25]:
!python scripts/train.py --config configs/train_colab_gpu.yaml --overrides model.trainable_backbone_layers=0 train.max_steps=2813

loading annotations into memory...
Done (t=2.54s)
creating index...
index created!
Config: {'data': {'ann_file': 'data/coco_subset/annotations/instances_train_subset.json', 'img_dir': 'data/coco_subset/images/train', 'val_ann_file': 'data/coco_subset/annotations/instances_val_subset.json', 'val_img_dir': 'data/coco_subset/images/val', 'n_images': 22500, 'img_size': 512, 'num_workers': 2}, 'model': {'backbone_name': 'resnet50', 'pretrained': True, 'trainable_backbone_layers': 0, 'cls_head_tap': 'fpn_p5'}, 'loss': {'det_cls': 1.0, 'det_box': 1.0, 'seg': 1.0, 'cls': 0.5}, 'train': {'device': 'cuda', 'batch_size': 8, 'epochs': 16, 'max_steps': 2813, 'lr': 0.0001, 'weight_decay': 0.0001, 'amp': True, 'seed': 42, 'log_every': 20, 'checkpoint_dir': 'checkpoints', 'run_name': 'colab_gpu', 'checkpoint_every_steps': 500}}
[step 0] classification=1.1579 bbox_regression=0.7766 seg_loss=4.5766 cls_loss=0.6965 total=6.8593
[step 20] classification=1.0699 bbox_regression=0.7197 seg_loss=4.0757 cls_lo

## 5. Evaluate + visualize a few predictions

In [27]:
!python scripts/eval.py --config configs/train_colab_gpu.yaml

usage: eval.py [-h] --config CONFIG --checkpoint CHECKPOINT
               [--results-csv RESULTS_CSV]
eval.py: error: the following arguments are required: --checkpoint


In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision.utils import draw_bounding_boxes

from mtl.config import load_config
from mtl.datasets.coco_multitask import CocoMultiTaskDataset
from mtl.engine.checkpoint import load_checkpoint
from mtl.models.multitask_model import MultiTaskModel
from mtl.utils.device import resolve_device

cfg = load_config("configs/train_colab_gpu.yaml")
device = resolve_device(cfg.train.device)
dataset = CocoMultiTaskDataset(cfg.data.val_ann_file, cfg.data.val_img_dir, img_size=cfg.data.img_size, train=False)

model = MultiTaskModel(
    det_num_classes=dataset.num_classes,
    seg_num_classes=dataset.num_classes + 1,
    cls_num_labels=dataset.num_classes,
).to(device)
load_checkpoint(model, optimizer=None, path="checkpoints/colab_gpu_epoch15.pt", map_location=str(device))
model.eval()

image, target = dataset[0]
with torch.no_grad():
    out = model(image.unsqueeze(0).to(device))

det = out["detections"][0]
keep = det["scores"] > 0.5
img_uint8 = ((image * 0.5 + 0.5) * 255).clamp(0, 255).byte()
drawn = draw_bounding_boxes(img_uint8, det["boxes"][keep].cpu())
plt.imshow(drawn.permute(1, 2, 0))
plt.title(f"top labels: {out['cls_pred'][0].topk(5).indices.tolist()}")
plt.show()

## 6. DINO omurgası (ikinci deney) + ResNet ile karşılaştırma
Aynı pipeline, backbone'da ResNet50+FPN yerine DINOv1 ViT-B/16 + Simple Feature Pyramid. `runs/results.csv` iki omurganın metriklerini biriktirir; `compare_results.py` markdown tablo basar.

In [ ]:
# --- DINO omurgası: tam koşu (çoklu oturum + resume'a dayanıklı) ---
# ResNet yolu aynen train_colab_gpu.yaml ile kalır; DINO sadece backbone'u değiştirir
# (configs/train_colab_dino.yaml). 12 GB VRAM için batch_size=4; sığmazsa aşağıya
# --overrides train.batch_size=2 ekle.
#
# Tam koşu (epochs=16). checkpoint_every_steps=500 ile her 500 adımda checkpoint Drive'a
# düşer. Oturum koparsa BİR SONRAKİ oturumda (aynı ya da paylaşımlı Drive'lı başka hesap)
# en son step-checkpoint'inden devam et -> artık --resume global ADIMdan devam eder,
# sıfırdan değil (scripts/train.py + engine/checkpoint.py). Tamamlanan epoch'lar atlanır.

# 1) İlk oturum - baştan başlat:
!python scripts/train.py --config configs/train_colab_dino.yaml

# 1b) Sonraki oturum(lar) - kaldığın step-checkpoint'inden devam et (en son .pt'yi yaz):
#     (checkpoints/ Drive'a symlink, o yüzden kopan oturumun checkpoint'i burada durur)
# !ls -t checkpoints/colab_dino_step*.pt | head -1     # en son checkpoint'i gör
# !python scripts/train.py --config configs/train_colab_dino.yaml --resume checkpoints/colab_dino_step3000.pt

# 2) Koşu bitince değerlendir (tam 16 epoch -> son checkpoint colab_dino_epoch15.pt;
#    erken durdurduysan en yüksek epoch/step'li .pt'yi yaz). results.csv'ye satır ekler:
!python scripts/eval.py --config configs/train_colab_dino.yaml --checkpoint checkpoints/colab_dino_epoch15.pt

# 3) Adil kıyas: ResNet'i de AYNI epoch'a kadar koşup eval et (elindeki .pt yolunu yaz):
# !python scripts/eval.py --config configs/train_colab_gpu.yaml --checkpoint checkpoints/colab_gpu_epoch15.pt

# 4) ResNet vs DINO karşılaştırma tablosu:
!python scripts/compare_results.py

## 7. DINOv2 omurgası (üçüncü deney) + karşılaştırma
Aynı pipeline, backbone'da DINOv2 ViT-B/14 (+4 register) + Simple Feature Pyramid (`configs/train_colab_dinov2.yaml`, `src/mtl/models/dinov2_backbone.py`). DINOv1 baseline'ı bozmamak için DINOv2 **ayrı dosyada**; head'ler/pipeline değişmez. `runs/results.csv` üç omurganın (ResNet / DINOv1 / DINOv2) metriklerini biriktirir.

**Not:** DINOv2 patch14 olduğu için config'te `img_size: 518` (14'e bölünebilir, modelin native çözünürlüğü). İlk çalıştırmada DINOv2 ağırlıkları HuggingFace'ten iner (birkaç sn). timm zaten section 1'de kurulu.

In [ ]:
# --- DINOv2 omurgası: tam koşu (çoklu oturum + resume'a dayanıklı) ---
# DINOv1 hücresiyle (section 6) birebir aynı akış, tek fark config: train_colab_dinov2.yaml
# (backbone dinov2_vitb14_reg, img_size=518). 12 GB VRAM için batch_size=4; sığmazsa
# --overrides train.batch_size=2 ekle. Hafif/hızlı istersen model.backbone_name=dinov2_vits14.

# 1) İlk oturum - baştan başlat:
!python scripts/train.py --config configs/train_colab_dinov2.yaml

# 1b) Sonraki oturum(lar) - en son step-checkpoint'inden devam et (checkpoints/ Drive'a symlink):
# !ls -t checkpoints/colab_dinov2_step*.pt | head -1     # en son checkpoint'i gör
# !python scripts/train.py --config configs/train_colab_dinov2.yaml --resume checkpoints/colab_dinov2_step3000.pt

# 2) Koşu bitince değerlendir (tam 16 epoch -> colab_dinov2_epoch15.pt). results.csv'ye satır ekler:
!python scripts/eval.py --config configs/train_colab_dinov2.yaml --checkpoint checkpoints/colab_dinov2_epoch15.pt

# 3) Üç omurga karşılaştırma tablosu (ResNet / DINOv1 / DINOv2 - hepsi results.csv'de):
!python scripts/compare_results.py